## Introduction
Cancer treatment is not equally effective for every patient. Tumours that look similar clinically can respond very differently to the same anticancer drug. One reason for this is the variety of genomic alterations present in the cells. Some of these alterations affect the biological pathways a drug targets and/or the mechanisms by which a cancer cell can survive drug treatment. If we can characterize the molecular features of a cancer cell, perhaps we can predit which drugs are likely to work on that cell. The Genomics of Drug Sensitivity in Cancer (GDSC) project has generated a large dataset combining:

- cancer cell lines
- genomic/molecular characteristics of those cell lines
- measurement of their responses to anticancer compounds

This makes the GDSC well suited to investigating the relationship between genomic features and drug response. 

Because the genomic feature space is enormous, Machine Learning (ML) provides methods for efficiently learning the relationships between genomic features and drug response. Importantly, it is uncertain whether genomic information actually contains enough signal to predict drug response.

**Question:** Can anticancer drug response be predicted from genomic features using machine-learning models trained on GDSC data, and which genomic features contribute most to predictive performance?

This question can be broken down into sevearal sub-questions:
1. **Predictability:** Is anticancer drug response predictable from genomic features?
2. **Model Performance:** Which machine-learning models provice the most useful predictive performance? Which models are most efficient? Is there an intersection between performance and efficiency?
3. **Generalization:** Does the predictive performance of the models generalize to unseen data?
4. **Biological Interpretation:** Which genomic features contribute most to predictive performance? Can we interpret the models to understand the biological mechanisms underlying drug response?
5. **Drug Specificity:** Are there specific drugs for which genomic features are particularly predictive of response? Conversely, are there drugs for which genomic features provide little predictive power?

**Null Hypothesis:** Genomic features do not contain enough information to predict anticancer drug response, with the methods explored.

**Alternative Hypothesis:** Genomic features exist that can provide predictive information about anticancer drug response, and the machine-learning models exploured can be trained to leverage this information effectively.

## Data Ingestion
### Overview

The data ingestion phase established a reproducible pipeline for acquiring and combining the datasets required to investigate whether anticancer drug response can be predicted from genomic features.

The analysis uses GDSC release 8.4, released July 24, 2022, for drug-response measurements and cell-line metadata. Gene-expression features are obtained separately from the COSMIC Cell Lines Project, version 104, using the GRCh38 release. The two sources use different identifiers, so an explicit mapping procedure was required before the datasets could be joined.

The goal of this phase was to acquire the original source data without modifying it, validate its structure, and construct a reproducible representation suitable for subsequent preprocessing.

### GDSC response data

The primary drug-response data were obtained from the official Sanger CancerRxGene GDSC release 8.4. The release contains two fitted single-agent response datasets:

- GDSC1 fitted dose-response data
- GDSC2 fitted dose-response data

Both datasets were downloaded and retained rather than selecting one at the ingestion stage. They were concatenated into a single response table while retaining the DATASET field so that the origin of each observation remains identifiable.

The response data contain, among other fields:

- `COSMIC_ID`
- `CELL_LINE_NAME`
- `DRUG_NAME`
- `AUC`
- `LN_IC50`
- `DATASET`

Both AUC and LN_IC50 were retained at this stage. Selection of the final response variable is a modelling decision and therefore belongs to the preprocessing/analysis phase rather than data acquisition.

### GDSC cell-line metadata

The GDSC release also provides Cell_Lines_Details.xlsx. This file was downloaded alongside the response data and used to attach biological metadata to the response observations.

The relevant metadata include:

- GDSC tissue of origin
- secondary tissue descriptor
- TCGA cancer type
- COSMIC identifier
- sample name
- availability indicators for WES, CNA, gene expression, and methylation

The tissue-of-origin field is particularly important because the research question is intended to support cancer-location-specific analyses rather than treating all cancer cell lines as a single homogeneous population.

The metadata are joined to the response data using the numeric COSMIC_ID supplied by GDSC.

### Gene-expression data

The GDSC release metadata indicate whether gene-expression data are available for each cell line, but the actual genomic feature matrix is not contained in the GDSC response files. Consequently, a second data source was required.

The selected genomic modality is gene expression from the COSMIC Cell Lines Project.

The downloaded COSMIC product is:

- Cell Lines Project Complete Gene Expression
- Version 104
- GRCh38
- Affymetrix Human Genome U219 Array

The data are supplied as a compressed TSV file contained within a TAR archive:

`CellLinesProject_CompleteGeneExpression_v104_GRCh38.tar`

containing:

`CellLinesProject_CompleteGeneExpression_v104_GRCh38.tsv.gz`

The expression file is initially in long format, with each row representing a cell-line/gene combination. The relevant fields are:

- COSMIC_SAMPLE_ID
- SAMPLE_NAME
- COSMIC_GENE_ID
- GENE_SYMBOL
- REGULATION
- Z_SCORE
- COSMIC_STUDY_ID

For this project, Z_SCORE is used as the expression measurement.

## Handling the COSMIC download

Unlike the historical GDSC release files, the current COSMIC download does not provide a permanent public URL. COSMIC generates a user-specific, time-limited signed URL.

Because the URL contains credentials and an expiration timestamp, it should not be committed to the repository. Instead, the current URL is supplied through an environment variable:

`COSMIC_LINK`

The ingestion code loads this value from `.env` using `python-dotenv`.

This approach allows the data-acquisition code to remain reproducible without embedding a user`s private, temporary COSMIC download URL in source control.

A complication encountered during development was that the signed URL could return HTTP 403 errors when it had expired. Refreshing the COSMIC download URL and placing the new value in .env resolved the problem. The already-downloaded archive is subsequently reused, so the signed URL is not required every time the dataset is loaded.

## Archive extraction

The COSMIC download is distributed as a TAR archive rather than directly as the expression TSV. The ingestion code therefore:

1. Obtains the signed URL from COSMIC_LINK.
2. Downloads the TAR archive to data/raw.
3. Inspects the archive contents.
4. Locates the expected expression file.
5. Extracts CellLinesProject_CompleteGeneExpression_v104_GRCh38.tsv.gz.
6. Retains the original compressed expression file for subsequent loading.

The archive and extracted expression file are treated as raw inputs rather than modified analytical datasets.

## Mapping GDSC to COSMIC

A major challenge was that GDSC and COSMIC do not use the same identifier for the cell lines.

GDSC provides a numeric `COSMIC_ID`, whereas the COSMIC expression dataset identifies samples using `COSMIC_SAMPLE_ID`, such as `COSS905985`.

The COSMIC sample file establishes the relationship between the COSMIC sample identifier and `SAMPLE_NAME`. The GDSC cell-line metadata also contains `Sample Name`.

Therefore, the mapping was established through the cell-line sample name:

`GDSC COSMIC_ID → GDSC Sample Name → COSMIC SAMPLE_NAME → COSMIC_SAMPLE_ID`

This was preferable to attempting to infer the relationship from the cell-line names in the drug-response table alone.

An empirical validation of this mapping was performed before incorporating expression data.

The comparison found:

- 1,002 GDSC sample names
- 1,020 COSMIC sample names
- 999 matching names
- 99.7% of GDSC names matched to COSMIC
- no COSMIC sample names mapped to multiple COSMIC sample IDs

Three GDSC names did not have a corresponding COSMIC sample name:

- `GT3TKB`
- `Hep 3B2_1-7`
- `TOTAL:`

`TOTAL:` was identified as a summary row rather than a biological cell line and was explicitly excluded from the mapping process.

The remaining unmatched names are retained as unmatched rather than being assigned an inferred identity.

## Expression data duplication

During integration, an apparent uniqueness problem was discovered in the COSMIC expression data.

The raw expression dataset contains approximately 17.5 million rows. Initial inspection showed approximately 1.86 million duplicate rows, indicating that the assumption that every `COSMIC_SAMPLE_ID`/`GENE_SYMBOL` combination was unique was incorrect.

Further investigation showed that the duplication was associated primarily with two COSMIC studies:

- COSU3
- COSU619

The duplication was not simply a matter of identical repeated records. Within `COSU619`, 11,688 sample/gene pairs occurred twice. Of those pairs, 9,814 had identical Z-scores, while 1,874 had slightly different Z-scores.

The two genes involved were:

- `ATXN7`
- `TMSB15B`

The differences were extremely small. For example, the largest observed differences were approximately 0.013 Z-score units.

This investigation was important because it demonstrated that blindly using `pivot()` with an assumed unique sample/gene key would be inappropriate. It also established that the duplicate records need to be understood as part of the COSMIC data structure rather than treated automatically as errors.

The ingestion phase therefore does not silently discard these observations. The duplicate structure is documented and must be resolved explicitly during preprocessing before constructing the final feature matrix.

## Long-to-wide transformation

The COSMIC expression source is distributed in long format:

`sample × gene → expression value`

Machine-learning models require a feature matrix in which each observation has a set of genomic features. The eventual representation therefore needs to be transformed into:

`cell line → gene 1, gene 2, gene 3, ...`

with the corresponding Z-score for each gene.

The ingestion code contains a preparation step for this transformation, but the duplicate observations discovered above mean that the final aggregation rule should be established during preprocessing rather than hidden inside the raw data-ingestion step.

This separation is intentional: ingestion should preserve the source data and expose structural problems, while preprocessing should document and apply the analytical decisions used to resolve them.

## Reproducibility and validation

The ingestion pipeline records the GDSC release information and the COSMIC expression product used. A manifest is written to the raw-data directory containing:

- GDSC release number
- GDSC release date
- source location
- source filenames
- COSMIC expression product
- COSMIC version
- genome build
- expected archive and expression filenames

Validation functions check that the expected GDSC response files, cell-line metadata, and COSMIC expression file exist and contain the required columns.

The pipeline is designed so that existing downloaded files are reused rather than downloaded repeatedly. This is particularly important for the COSMIC dataset because its download URL is temporary and user-specific.

## Result of the ingestion phase

At the end of data ingestion, the project has three principal raw data resources:

1. GDSC1 drug-response data
2. GDSC2 drug-response data
3. GDSC cell-line metadata
4. COSMIC v104 gene-expression data

The response data and metadata can be joined using `COSMIC_ID`. The expression data can be linked to the GDSC cell lines through the validated `SAMPLE_NAME` mapping.

No biological observations have yet been removed because of missing genomic features, no expression values have been imputed, no genes have been selected or filtered, and no machine-learning train/test split has been performed.

These decisions are deliberately deferred to the preprocessing stage.

The key output of ingestion is therefore *a validated, reproducible connection between drug-response observations, cancer/tissue metadata, and COSMIC gene-expression measurements*, while preserving enough information about the original sources and identifier mappings to make the subsequent analysis auditable.


In [3]:
from gdsc.data import download_gdsc, validate_gdsc, load_gdsc

files = download_gdsc("../data/raw")
print(files)

validate_gdsc("../data/raw")
print("Validation successful.")

{'gdsc1_response': PosixPath('../data/raw/GDSC1_fitted_dose_response_24Jul22.csv'), 'gdsc2_response': PosixPath('../data/raw/GDSC2_fitted_dose_response_24Jul22.csv'), 'cell_lines': PosixPath('../data/raw/Cell_Lines_Details.xlsx'), 'cosmic_expression_archive': PosixPath('../data/raw/CellLinesProject_CompleteGeneExpression_Tsv_v104_GRCh38.tar'), 'cosmic_expression': PosixPath('../data/raw/CellLinesProject_CompleteGeneExpression_v104_GRCh38.tsv.gz')}
Validation successful.


/home/ajharris/Projects/gdsc-project/venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [ ]:
import pandas as pd

expression = pd.read_csv(
    "../data/raw/CellLinesProject_CompleteGeneExpression_v104_GRCh38.tsv.gz",
    sep="\t",
    compression="gzip",
)

duplicates = expression[
    expression.duplicated(
        subset=["COSMIC_SAMPLE_ID", "GENE_SYMBOL"],
        keep=False,
    )
].copy()

print(f"Total expression rows: {len(expression):,}")
print(f"Duplicate rows: {len(duplicates):,}")

print(
    duplicates[
        [
            "COSMIC_SAMPLE_ID",
            "SAMPLE_NAME",
            "GENE_SYMBOL",
            "REGULATION",
            "Z_SCORE",
            "COSMIC_STUDY_ID",
        ]
    ].head(20)
)

In [ ]:
# 1. How many study IDs are represented among duplicate sample/gene pairs?
study_counts = (
    duplicates
    .groupby(["COSMIC_SAMPLE_ID", "GENE_SYMBOL"])["COSMIC_STUDY_ID"]
    .nunique()
)

print(study_counts.describe())
print(study_counts.value_counts().sort_index())

count    927960.000000
mean          1.988103
std           0.108423
min           1.000000
25%           2.000000
50%           2.000000
75%           2.000000
max           2.000000
Name: COSMIC_STUDY_ID, dtype: float64
COSMIC_STUDY_ID
1     11040
2    916920
Name: count, dtype: int64


In [ ]:
# 1. How many study IDs are represented among duplicate sample/gene pairs?
study_counts = (
    duplicates
    .groupby(["COSMIC_SAMPLE_ID", "GENE_SYMBOL"])["COSMIC_STUDY_ID"]
    .nunique()
)

print(study_counts.describe())
print(study_counts.value_counts().sort_index())

count    927960.000000
mean          1.988103
std           0.108423
min           1.000000
25%           2.000000
50%           2.000000
75%           2.000000
max           2.000000
Name: COSMIC_STUDY_ID, dtype: float64
COSMIC_STUDY_ID
1     11040
2    916920
Name: count, dtype: int64


In [ ]:
# 2. Are there duplicates even within the same study?
within_study = duplicates.duplicated(
    subset=[
        "COSMIC_SAMPLE_ID",
        "GENE_SYMBOL",
        "COSMIC_STUDY_ID",
    ],
    keep=False,
)

print(f"Duplicate rows within sample/gene/study: {within_study.sum():,}")

Duplicate rows within sample/gene/study: 24,672


In [ ]:
# 3. How different are the values across studies?
variation = (
    duplicates
    .groupby(["COSMIC_SAMPLE_ID", "GENE_SYMBOL"])["Z_SCORE"]
    .agg(["count", "nunique", "min", "max"])
)

print(variation.head(20))
print(
    "Pairs with different Z-scores:",
    (variation["nunique"] > 1).sum()
)

                              count  nunique    min    max
COSMIC_SAMPLE_ID GENE_SYMBOL                              
COSS1240121      ABCF2            2        1 -0.436 -0.436
                 ATXN7            2        2 -0.197 -0.193
                 CCDC39           2        1  0.905  0.905
                 HSPA14           2        1 -0.506 -0.506
                 IGF2             2        1  0.706  0.706
                 MATR3            2        1 -1.615 -1.615
                 PDE11A           2        1  2.291  2.291
                 PRSS50           2        1  0.226  0.226
                 SCO2             2        1  1.698  1.698
                 SOD2             2        1  1.543  1.543
                 TBCE             2        1 -0.407 -0.407
                 TMSB15B          2        2 -1.056 -1.050
COSS1240122      ABCF2            2        1  0.017  0.017
                 ATXN7            2        2 -0.958 -0.954
                 CCDC39           2        1 -1.588 -1.5

We are investigating the merge of the GDSC set with the COSMIC set, it isn't as straightforward as it initially seemed.

In [ ]:
# Diagnose the remaining within-study duplicates in COSU619.
# Uses the existing `expression` DataFrame; does not reread the TSV.

cosu619 = expression[
    expression["COSMIC_STUDY_ID"].eq("COSU619")
].copy()

print(f"COSU619 rows: {len(cosu619):,}")

# Identify sample/gene pairs that occur more than once within COSU619.
pair_counts = (
    cosu619
    .groupby(
        ["COSMIC_SAMPLE_ID", "GENE_SYMBOL"],
        sort=False,
        observed=True,
    )
    .size()
)

duplicate_pairs = pair_counts[pair_counts > 1]

print(f"Within-COSU619 duplicate sample/gene pairs: {len(duplicate_pairs):,}")
print(f"Rows belonging to those pairs: {duplicate_pairs.sum():,}")

# Extract only the affected rows.
cosu619_duplicates = cosu619.merge(
    duplicate_pairs.rename("PAIR_COUNT").reset_index(),
    on=["COSMIC_SAMPLE_ID", "GENE_SYMBOL"],
    how="inner",
)

# Summarize whether duplicate records have identical or different Z-scores.
zscore_summary = (
    cosu619_duplicates
    .groupby(
        ["COSMIC_SAMPLE_ID", "GENE_SYMBOL"],
        sort=False,
        observed=True,
    )["Z_SCORE"]
    .agg(
        COUNT="size",
        UNIQUE_Z_SCORES="nunique",
        MIN_Z="min",
        MAX_Z="max",
    )
)

print("\nDuplicate-pair Z-score summary:")
print(zscore_summary["UNIQUE_Z_SCORES"].value_counts().sort_index())

print(
    "\nPairs with different Z-scores:",
    (zscore_summary["UNIQUE_Z_SCORES"] > 1).sum(),
)

# Show representative examples of genuinely different values.
different = zscore_summary[
    zscore_summary["UNIQUE_Z_SCORES"] > 1
].reset_index()

if not different.empty:
    print("\nExamples with different Z-scores:")
    print(
        cosu619_duplicates.merge(
            different[
                ["COSMIC_SAMPLE_ID", "GENE_SYMBOL"]
            ],
            on=["COSMIC_SAMPLE_ID", "GENE_SYMBOL"],
            how="inner",
        )[
            [
                "COSMIC_SAMPLE_ID",
                "SAMPLE_NAME",
                "GENE_SYMBOL",
                "Z_SCORE",
                "COSMIC_STUDY_ID",
                "PAIR_COUNT",
            ]
        ].head(30).to_string(index=False)
    )
else:
    print("\nNo COSU619 duplicate pairs have different Z-scores.")

COSU619 rows: 16,550,208
Within-COSU619 duplicate sample/gene pairs: 11,688
Rows belonging to those pairs: 23,376

Duplicate-pair Z-score summary:
UNIQUE_Z_SCORES
1    9814
2    1874
Name: count, dtype: int64

Pairs with different Z-scores: 1874

Examples with different Z-scores:
COSMIC_SAMPLE_ID SAMPLE_NAME GENE_SYMBOL  Z_SCORE COSMIC_STUDY_ID  PAIR_COUNT
      COSS949165       EW-16     TMSB15B    0.959         COSU619           2
     COSS1298538 RERF-LC-Sq1     TMSB15B   -0.377         COSU619           2
      COSS909751        SW48     TMSB15B   -1.497         COSU619           2
      COSS753581 LB373-MEL-D     TMSB15B   -0.311         COSU619           2
     COSS1240192    NCI-H841     TMSB15B   -0.121         COSU619           2
      COSS910937    COR-L279     TMSB15B    1.024         COSU619           2
     COSS1659819     OCI-LY7     TMSB15B    0.833         COSU619           2
      COSS946358          YT     TMSB15B   -0.452         COSU619           2
     COSS1524419 

In [ ]:
# Which genes account for the COSU619 duplicate pairs with
# different Z-scores?

different_genes = (
    different
    .groupby("GENE_SYMBOL")
    .size()
    .sort_values(ascending=False)
)

print(different_genes.head(30).to_string())

print(
    f"\nNumber of genes involved: {different_genes.size:,}"
)

# How many duplicate pairs have differences that are actually substantial?

different_values = (
    cosu619_duplicates
    .groupby(
        ["COSMIC_SAMPLE_ID", "GENE_SYMBOL"],
        sort=False,
        observed=True,
    )["Z_SCORE"]
    .agg(["min", "max"])
)

different_values["ABS_DIFF"] = (
    different_values["max"] - different_values["min"]
)

print(different_values["ABS_DIFF"].describe())

print(
    "\nLargest Z-score differences:"
)

print(
    different_values
    .sort_values("ABS_DIFF", ascending=False)
    .head(20)
    .to_string()
)

GENE_SYMBOL
ATXN7      974
TMSB15B    900

Number of genes involved: 2
count    11688.000000
mean         0.000628
std          0.001620
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max          0.013000
Name: ABS_DIFF, dtype: float64

Largest Z-score differences:
                                min    max  ABS_DIFF
COSMIC_SAMPLE_ID GENE_SYMBOL                        
COSS910905       TMSB15B     -2.665 -2.652     0.013
COSS724866       TMSB15B     -2.136 -2.124     0.012
COSS1298134      TMSB15B     -2.242 -2.230     0.012
COSS907318       TMSB15B     -2.186 -2.175     0.011
COSS949175       TMSB15B      3.053  3.064     0.011
COSS1290810      TMSB15B     -2.186 -2.175     0.011
COSS753569       TMSB15B     -2.035 -2.024     0.011
COSS905968       TMSB15B     -1.939 -1.928     0.011
COSS1298533      TMSB15B     -2.075 -2.064     0.011
COSS753588       TMSB15B      3.174  3.185     0.011
COSS687780       TMSB15B     -1.924 -1.913     0.011
COS

In [ ]:
data = load_gdsc(
    data_dir="../data/raw",
    include_metadata=True,
    include_expression=True,
)

print(data.shape)

/home/ajharris/Projects/gdsc-project/venv/lib/python3.12/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


ValueError: COSMIC expression contains duplicate sample/gene pairs. Found 927,960 duplicated pairs.

### Initial dataset characterization

The following summaries describe the loaded release before any preprocessing or analytical filtering. Tissue and cancer labels are retained as metadata; no cancer type is selected at this stage.

In [ ]:
tissue_summary = (
    data.groupby("TISSUE_OF_ORIGIN")
    .agg(
        observations=("COSMIC_ID", "size"),
        cell_lines=("COSMIC_ID", "nunique"),
        drugs=("DRUG_ID", "nunique"),
    )
    .sort_values("observations", ascending=False)
)
tissue_summary

,observations,cell_lines,drugs
TISSUE_OF_ORIGIN,,,
lung_NSCLC,64499,108,621
urogenital_system,61288,104,621
leukemia,50008,84,621
aero_dig_tract,45354,77,621
lymphoma,40948,69,621
lung_SCLC,33558,63,621
skin,33253,58,621
nervous_system,32794,55,621
breast,31021,52,621


### Genomic availability and quality checks

The response files contain no genomic feature matrix. `Cell_Lines_Details.xlsx` records whether WES, CNA, gene expression, and methylation assays are available for each model; the actual feature files and their identifiers must be selected and joined in a later research-design step. The checks below report raw-data limitations without removing observations.

### Data-to-Preprocessing checkpoint

This analysis uses official Sanger CancerRxGene GDSC release 8.4 (24 July 2022), combining GDSC1 and GDSC2 fitted single-agent response files with `Cell_Lines_Details.xlsx`. Each observation represents a cell-line/drug response record and retains COSMIC model ID, tissue of origin, and cancer metadata. AUC and LN_IC50 are available and complete in the loaded release; neither is selected as the final target yet.

The release metadata reports WES, CNA, gene-expression, and methylation availability flags, but does not include the genomic feature matrices themselves. Cancer type is missing for a substantial subset of observations, while tissue labels are retained. Before preprocessing, the project still needs to choose a genomic modality and source, define eligibility and missingness rules, select a response measure, and decide how cancer types will be compared. No observations have been imputed, filtered, normalized, or split at this checkpoint.

### Reproducible preprocessing interface

The preprocessing implementation lives in `gdsc.preprocessing`, so the notebook does not duplicate hidden cleaning steps. It requires a selected genomic matrix with one verified shared cell-line identifier (normally `COSMIC_ID`). The downloaded release inspected above does not supply that matrix: its availability flags only indicate whether an assay exists for a model. Therefore executing a real join here would falsely imply that a genomic modality has been selected.

Once a feature matrix is obtained, call `preprocess_gdsc` with the documented study thresholds. The returned `X` will contain genomic columns only; `y` will be the explicit AUC or LN_IC50 choice; and tissue, cancer type, drug, and identifiers will remain in `metadata`. The function records every row/feature count in `prepared.info`. It does not fit an imputer or scaler. Any later transformer must be fit on the training split only.

The next cell is deliberately a readiness check, not a call to `preprocess_gdsc`: no genomic feature matrix is currently available. This checkpoint intentionally stops before a train/test split and model training.

### Reproducible preprocessing interface

The preprocessing implementation lives in `gdsc.preprocessing`, so the notebook does not duplicate hidden cleaning steps. It requires a selected genomic matrix with one verified shared cell-line identifier (normally `COSMIC_ID`). The downloaded release inspected above does not supply that matrix: its availability flags only indicate whether an assay exists for a model. Therefore executing a real join here would falsely imply that a genomic modality has been selected.

Once a feature matrix is obtained, call `preprocess_gdsc` with the documented study thresholds. The returned `X` will contain genomic columns only; `y` will be the explicit AUC or LN_IC50 choice; and tissue, cancer type, drug, and identifiers will remain in `metadata`. The function records every row/feature count in `prepared.info`. It does not fit an imputer or scaler. Any later transformer must be fit on the training split only.

No preprocessing call is shown here because no genomic feature matrix is currently available. This checkpoint intentionally stops before a train/test split and model training.